# Notas — Aula 6: Encapsulamento e properties

Marco: o robô que já era `class Robo` (Aula 5) ganha proteção de verdade. `x`, `y` e
`bateria` deixam de ser atributos soltos — passam a ser **properties**, que se leem
como atributo mas escondem um método validado por trás. Antes de chegar lá, vemos o
porquê: a convenção `_atributo` (comunica, não protege) e a diferença real entre um e
dois underscores.

> ⚠️ **Antes de começar:** rode todas as células a partir do topo (**Run All**) — cada
> seção define suas próprias classes; se aparecer `NameError`, é sinal de que uma
> célula anterior ainda não rodou nesta sessão do kernel.

## Convenção `_atributo` — esconder por acordo

Um atributo com `_` na frente (`_nivel`) continua sendo um atributo Python comum —
nada impede leitura ou escrita de fora. O `_` é só um **pedido**: "isto é
implementação interna, não construa nada em cima". Quem quer proteção de verdade
precisa de um getter/setter (à moda antiga) ou, melhor ainda, de `@property` — que
vem duas seções abaixo. No robô, `bateria` era exatamente esse tipo de atributo
"pedindo" para não ser tocado direto.

In [1]:
class Bateria:
    def __init__(self, nivel=100):
        self.nivel = nivel

b = Bateria()
b.nivel = -999
print(b.nivel)

-999


In [2]:
class BateriaConvencao:
    def __init__(self, nivel=100):
        self._nivel = nivel

    def get_nivel(self):
        return self._nivel

    def set_nivel(self, valor):
        if 0 <= valor <= 100:
            self._nivel = valor
        else:
            raise ValueError(f"nivel={valor} fora de [0, 100]")

b2 = BateriaConvencao()
try:
    b2.set_nivel(150)
except ValueError as erro:
    print(erro)

nivel=150 fora de [0, 100]


### Sua vez

Complete `set_alcance(self, valor)` da classe `Sensor`: levante `ValueError` se
`valor` estiver fora de `[0, 100]`; senão, guarde em `self._alcance`.

*Dica: mesma estrutura de `BateriaConvencao.set_nivel` acima.*

In [3]:
class Sensor:
    def __init__(self, alcance=50):
        self._alcance = alcance

    def get_alcance(self):
        return self._alcance

    def set_alcance(self, valor):
        if 0 <= valor <= 100:
            self._alcance = valor
        else:
            raise ValueError(f"alcance={valor} fora de [0, 100]")


s = Sensor()
s.set_alcance(80)
print(s.get_alcance())

80


## `_atributo` vs. `__atributo` — convenção e mecanismo real

Um underscore é convenção; **dois** underscores o Python de fato transforma —
*name mangling*: `self.__motor` dentro de `Carro` vira, na prática, o atributo
`_Carro__motor`. É por isso que `carro.__motor` (de fora) dá erro, mas
`carro._Carro__motor` funciona. A comunidade Python majoritariamente usa só `_`,
mesmo para dado bem interno — `__` resolve um problema específico de herança que só
aparece mais adiante no curso.

In [4]:
class Motor:
    def __init__(self, potencia):
        self.potencia = potencia

class Carro:
    def __init__(self, modelo, potencia_motor):
        self.modelo = modelo
        self.__motor = Motor(potencia_motor)

carro = Carro("HRV", 150)
print(vars(carro))

{'modelo': 'HRV', '_Carro__motor': <__main__.Motor object at 0x104175940>}


In [5]:
try:
    print(carro.__motor)
except AttributeError as erro:
    print(erro)

print(carro._Carro__motor.potencia)

'Carro' object has no attribute '__motor'
150


### Sua vez

Complete `depositar(self, valor)` da classe `ContaSecreta`: aumente `self.__saldo`
em `valor` e devolva o novo saldo. (Dentro da própria classe, `__saldo` funciona
normalmente — o mangling só afeta acesso de *fora*.)

In [6]:
class ContaSecreta:
    def __init__(self, saldo=0):
        self.__saldo = saldo

    def depositar(self, valor):
        self.__saldo += valor
        return self.__saldo


c = ContaSecreta()
print(c.depositar(50))

50


## `@property`: a proteção que se lê como atributo

`@property` transforma um método num getter que se lê como atributo comum
(`objeto.nome`, sem parênteses); `@nome.setter` define o que roda quando alguém
escreve `objeto.nome = valor`. As duas coisas juntas dão a proteção de um
getter/setter à moda antiga com a leitura natural de um atributo.

In [7]:
class Termometro:
    def __init__(self, celsius=0):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, valor):
        if valor < -273.15:
            raise ValueError(f"celsius={valor} é abaixo do zero absoluto")
        self._celsius = valor


t = Termometro(20)
print(t.celsius)
t.celsius = 25
print(t.celsius)

20
25


### Sua vez

Complete o setter de `metros` da classe `Distancia`: levante `ValueError` se
`valor < 0`; senão, guarde em `self._metros`.

In [8]:
class Distancia:
    def __init__(self, metros=0):
        self._metros = metros

    @property
    def metros(self):
        return self._metros

    @metros.setter
    def metros(self, valor):
        if valor < 0:
            raise ValueError(f"metros={valor} não pode ser negativo")
        self._metros = valor


d = Distancia()
d.metros = 10
print(d.metros)

10


## Aplicando no robô: coordenada que recusa sair da grade

Mesma receita do `Termometro`, agora validando a invariante do robô: `x` fora de
`[0, LADO_GRADE - 1]` levanta `ValueError` em vez de aceitar em silêncio.

### Sua vez

Complete o setter de `x` da classe `Robo`: levante `ValueError` se `valor` estiver
fora de `[0, LADO_GRADE - 1]`; senão, guarde em `self._x`.

In [9]:
class Robo:
    LADO_GRADE = 10

    def __init__(self, x=0, y=0):
        self._x = x
        self._y = y

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, valor):
        if not (0 <= valor < Robo.LADO_GRADE):
            raise ValueError(f"x={valor} sai da grade (0 a {Robo.LADO_GRADE - 1})")
        self._x = valor


r = Robo()
try:
    r.x = 999
except ValueError as erro:
    print(erro)
print(r.x)

x=999 sai da grade (0 a 9)
0


## Bateria: presa nos limites em vez de recusada

Nem toda validação recusa. Para a bateria, prender o valor dentro de `[0, 100]`
(*clamp*, com `max`/`min`) é mais adequado do que levantar erro — bateria em 0 é
estado normal, não bug.

### Sua vez

Complete o setter de `bateria` da classe `Robo`: prenda `valor` entre `0` e `100`
(sem levantar erro) e guarde em `self._bateria`.

In [10]:
class Robo:
    def __init__(self, bateria=50):
        self._bateria = bateria

    @property
    def bateria(self):
        return self._bateria

    @bateria.setter
    def bateria(self, valor):
        self._bateria = max(0, min(100, valor))


r = Robo()
r.bateria = 150
print(r.bateria)

100


## Propriedade somente leitura

Sem `@nome.setter`, a property não aceita escrita — o Python recusa de cara, sem
rodar nada.

In [11]:
class RoboComPosicao:
    def __init__(self, x=0, y=0):
        self._x = x
        self._y = y

    @property
    def posicao(self):
        return (self._x, self._y)


rp = RoboComPosicao(3, 4)
print(rp.posicao)

try:
    rp.posicao = (5, 5)
except AttributeError as erro:
    print(erro)

(3, 4)
property 'posicao' of 'RoboComPosicao' object has no setter


### Sua vez

Complete a property `posicao` da classe `Robo` (sem setter — somente leitura):
devolva a tupla `(self._x, self._y)`.

In [12]:
class Robo:
    def __init__(self, x=0, y=0):
        self._x = x
        self._y = y

    @property
    def posicao(self):
        return (self._x, self._y)


r = Robo(3, 4)
print(r.posicao)

(3, 4)


## Para aprofundar

- Encapsulamento e a convenção de underscore — PEP 8: https://peps.python.org/pep-0008/#descriptive-naming-styles
- `@property` (fundamentos) — Tutorial oficial: https://docs.python.org/3/library/functions.html#property
- Name mangling (`__atributo`) — documentação oficial: https://docs.python.org/3/tutorial/classes.html#private-variables
- Getters e setters em Python — Real Python: https://realpython.com/python-getter-setter/